# Shape Mismatch Leak — MatthewZou9419/GACNN

**Smell (`GA.py` lines 58, 70):** `Flatten()` is applied directly before `Dense(120)` inside the CNN model. After a sequence of Conv2D and MaxPooling layers, the spatial feature map is flattened into a large vector, creating a massive weight matrix connecting every pixel-channel pair to every neuron in the dense layer.

**Fix:** Replace `Flatten() → Dense(120)` with `GlobalAveragePooling2D() → Dense(120)`. GAP collapses spatial dimensions to a single value per channel, drastically reducing the weight matrix and memory footprint.

In [ ]:
!pip install -q codecarbon

In [ ]:
import codecarbon, os, json
_dir = os.path.join(os.path.dirname(codecarbon.__file__), 'data', 'private_infra')
os.makedirs(_dir, exist_ok=True)
_p = os.path.join(_dir, 'nordic_emissions.json')
if not os.path.exists(_p):
    with open(_p, 'w') as f: json.dump({}, f)
print('CodeCarbon ready')

In [ ]:
import gc, os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten,
                                      GlobalAveragePooling2D, Dense, Dropout)
from tensorflow.keras.callbacks import EarlyStopping
from codecarbon import EmissionsTracker

os.makedirs('results', exist_ok=True)
N_RUNS = 10
EPOCHS = 5
BATCH  = 64

(x_tr, y_tr), (x_te, y_te) = cifar10.load_data()
x_tr = x_tr.astype('float32') / 255.0
x_te = x_te.astype('float32') / 255.0
y_tr_oh = to_categorical(y_tr, 10)
y_te_oh = to_categorical(y_te, 10)
print('CIFAR-10 loaded:', x_tr.shape, x_te.shape)

In [ ]:
def build_smelly_gacnn():
    """MatthewZou9419 GA.py — SMELL: Flatten() then Dense(120) at lines 58, 70"""
    model = Sequential([
        Conv2D(6,  (5,5), activation='relu', input_shape=(32,32,3)),
        MaxPooling2D((2,2)),
        Conv2D(16, (5,5), activation='relu'),
        MaxPooling2D((2,2)),
        # ❌ SMELL (lines 58, 70): Flatten then Dense — large weight matrix
        Flatten(),
        Dense(120, activation='relu', use_bias=False),
        Dense(84,  activation='relu', use_bias=False),
        Dense(10,  activation='softmax', use_bias=False),
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def build_clean_gacnn():
    """Fixed: GlobalAveragePooling2D replaces Flatten()"""
    model = Sequential([
        Conv2D(6,  (5,5), activation='relu', input_shape=(32,32,3)),
        MaxPooling2D((2,2)),
        Conv2D(16, (5,5), activation='relu'),
        MaxPooling2D((2,2)),
        # ✅ FIX: GlobalAveragePooling2D — spatial average, compact vector
        GlobalAveragePooling2D(),
        Dense(120, activation='relu', use_bias=False),
        Dense(84,  activation='relu', use_bias=False),
        Dense(10,  activation='softmax', use_bias=False),
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

print('Model builders ready')
m = build_smelly_gacnn(); m.summary(); del m
m = build_clean_gacnn();  m.summary(); del m

## BEFORE — Smell Active (Flatten → Dense(120))

In [ ]:
results_before = []
es = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

for run in range(1, N_RUNS + 1):
    print(f'\n[BEFORE] Run {run}/{N_RUNS}')
    K.clear_session(); gc.collect()
    tracker = EmissionsTracker(
        project_name=f'GACNN_before_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()
    model = build_smelly_gacnn()
    history = model.fit(x_tr, y_tr_oh, epochs=EPOCHS, batch_size=BATCH,
                        validation_data=(x_te, y_te_oh), callbacks=[es], verbose=1)
    _, acc = model.evaluate(x_te, y_te_oh, verbose=0)
    tracker.stop()
    em = tracker.final_emissions_data
    results_before.append({
        'run': run, 'accuracy': round(float(acc),4),
        'epochs_run': len(history.history['loss']),
        'energy_kWh': round(em.energy_consumed,8),
        'cpu_energy_kWh': round(em.cpu_energy,8),
        'gpu_energy_kWh': round(em.gpu_energy,8),
        'ram_energy_kWh': round(em.ram_energy,8),
        'emissions_kgCO2': round(em.emissions,8),
        'duration_s': round(em.duration,2),
    })
    print(f'  acc={acc:.4f}  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')
    del model; gc.collect()

df_before = pd.DataFrame(results_before)
df_before.to_csv('results/MatthewZou9419_GACNN_before.csv', index=False)
print('\n--- BEFORE means ---')
print(df_before.mean(numeric_only=True))
print('Saved results/MatthewZou9419_GACNN_before.csv')

## AFTER — Smell Fixed (GlobalAveragePooling2D → Dense)

In [ ]:
results_after = []

for run in range(1, N_RUNS + 1):
    print(f'\n[AFTER] Run {run}/{N_RUNS}')
    K.clear_session(); gc.collect()
    tracker = EmissionsTracker(
        project_name=f'GACNN_after_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()
    model = build_clean_gacnn()
    history = model.fit(x_tr, y_tr_oh, epochs=EPOCHS, batch_size=BATCH,
                        validation_data=(x_te, y_te_oh), callbacks=[es], verbose=1)
    _, acc = model.evaluate(x_te, y_te_oh, verbose=0)
    tracker.stop()
    em = tracker.final_emissions_data
    results_after.append({
        'run': run, 'accuracy': round(float(acc),4),
        'epochs_run': len(history.history['loss']),
        'energy_kWh': round(em.energy_consumed,8),
        'cpu_energy_kWh': round(em.cpu_energy,8),
        'gpu_energy_kWh': round(em.gpu_energy,8),
        'ram_energy_kWh': round(em.ram_energy,8),
        'emissions_kgCO2': round(em.emissions,8),
        'duration_s': round(em.duration,2),
    })
    print(f'  acc={acc:.4f}  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')
    del model; gc.collect()

df_after = pd.DataFrame(results_after)
df_after.to_csv('results/MatthewZou9419_GACNN_after.csv', index=False)
print('\n--- AFTER means ---')
print(df_after.mean(numeric_only=True))
print('Saved results/MatthewZou9419_GACNN_after.csv')

In [ ]:
print('\n=== SUMMARY: MatthewZou9419/GACNN ===')
print(f"BEFORE avg energy : {df_before['energy_kWh'].mean():.6f} kWh")
print(f"AFTER  avg energy : {df_after['energy_kWh'].mean():.6f} kWh")
print(f"BEFORE avg CO2    : {df_before['emissions_kgCO2'].mean():.6f} kg")
print(f"AFTER  avg CO2    : {df_after['emissions_kgCO2'].mean():.6f} kg")
print(f"BEFORE avg time   : {df_before['duration_s'].mean():.1f} s")
print(f"AFTER  avg time   : {df_after['duration_s'].mean():.1f} s")